In [14]:
import os
from pathlib import Path

import pandas as pd
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from transformers import AutoTokenizer, AutoModel

In [15]:
# =========================================================
# PATHS
# =========================================================
PROJECT_DIR = Path(r"C:\Users\Home\Desktop\projet-Ai")
SPLIT_DIR = PROJECT_DIR / "data" / "Split Dataset"
IMAGE_DIR = PROJECT_DIR / "data" / "Labelled Images"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = SPLIT_DIR / "Training_meme_dataset.csv"
VAL_CSV = SPLIT_DIR / "validation_meme_dataset.csv"
TEST_CSV = SPLIT_DIR / "Testing_meme_dataset.csv"

MODEL_PATH = MODEL_DIR / "meme-classification-multimodal_model_xpu.pth"

In [16]:
# =========================================================
# SETTINGS
# =========================================================
IMAGE_COL = "image_name"
TEXT_COL = "sentence"
LABEL_COL = "label"

BATCH_SIZE = 8
NUM_EPOCHS = 6
LEARNING_RATE = 2e-5
NUM_WORKERS = 0
MAX_TEXT_LEN = 128

TEXT_MODEL_NAME = "distilbert-base-uncased"

In [17]:
# =========================================================
# DEVICE
# =========================================================
if hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: xpu


In [18]:
# =========================================================
# LABEL CLEANING
# =========================================================
def clean_label(x):
    x = str(x).strip().lower()

    if x in ["non-offensiv", "non-offensive", "non offensive", "nonoffensive"]:
        return "non_offensive"
    if x in ["offensive", "offensiv"]:
        return "offensive"

    raise ValueError(f"Unexpected label found: {x}")


label_to_id = {
    "non_offensive": 0,
    "offensive": 1
}

id_to_label = {
    0: "non_offensive",
    1: "offensive"
}

In [19]:
# =========================================================
# LOAD CSV FILES
# =========================================================
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

for df in [train_df, val_df, test_df]:
    df[LABEL_COL] = df[LABEL_COL].apply(clean_label)
    df[LABEL_COL] = df[LABEL_COL].map(label_to_id)
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain label counts:")
print(train_df[LABEL_COL].value_counts().sort_index())

Train shape: (445, 3)
Val shape: (149, 3)
Test shape: (149, 3)

Train label counts:
label
0    258
1    187
Name: count, dtype: int64


In [20]:
# =========================================================
# CHECK IMAGE FILES
# =========================================================
def image_exists(image_name):
    image_path = IMAGE_DIR / str(image_name)
    return image_path.exists()

# verify images and optionally remove missing entries
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = df[~df[IMAGE_COL].apply(image_exists)]
    print(f"\nMissing images in {name}: {len(missing)}")
    if len(missing) > 0:
        # show a sample of the missing filenames
        print(missing[[IMAGE_COL]].head(10))
        # drop rows referencing missing files so downstream logic can continue
        df.drop(missing.index, inplace=True)
        print(f"Dropped {len(missing)} rows from {name} split")
        # you can also log or save `missing` if you wish to investigate further



Missing images in train: 0

Missing images in val: 1
    image_name
8  KbTk7Rq.png
Dropped 1 rows from val split

Missing images in test: 0


In [21]:
# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [22]:
# =========================================================
# TOKENIZER
# =========================================================
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)


# =========================================================
# DATASET
# =========================================================
class MemeMultimodalDataset(Dataset):
    def __init__(self, df, image_dir, tokenizer, max_text_len, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_name = str(self.df.loc[idx, IMAGE_COL])
        text = str(self.df.loc[idx, TEXT_COL])
        label = int(self.df.loc[idx, LABEL_COL])

        image_path = self.image_dir / image_name
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_text_len,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)

        return {
            "image": image,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": torch.tensor(label, dtype=torch.long)
        }

In [23]:
# =========================================================
# DATALOADERS
# =========================================================
train_dataset = MemeMultimodalDataset(
    train_df, IMAGE_DIR, tokenizer, MAX_TEXT_LEN, train_transform
)
val_dataset = MemeMultimodalDataset(
    val_df, IMAGE_DIR, tokenizer, MAX_TEXT_LEN, eval_transform
)
test_dataset = MemeMultimodalDataset(
    test_df, IMAGE_DIR, tokenizer, MAX_TEXT_LEN, eval_transform
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

In [24]:
# =========================================================
# MULTIMODAL MODEL
# =========================================================
class MultimodalMemeClassifier(nn.Module):
    def __init__(self, text_model_name, num_classes=2):
        super().__init__()

        # image encoder
        self.image_encoder = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        image_feature_dim = self.image_encoder.fc.in_features
        self.image_encoder.fc = nn.Identity()

        # text encoder
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        text_feature_dim = self.text_encoder.config.hidden_size

        # projection layers
        self.image_proj = nn.Sequential(
            nn.Linear(image_feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.text_proj = nn.Sequential(
            nn.Linear(text_feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        image_features = self.image_encoder(image)
        image_features = self.image_proj(image_features)

        text_outputs = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        text_features = text_outputs.last_hidden_state[:, 0, :]
        text_features = self.text_proj(text_features)

        fused = torch.cat([image_features, text_features], dim=1)
        logits = self.classifier(fused)
        return logits


model = MultimodalMemeClassifier(TEXT_MODEL_NAME, num_classes=2)
model = model.to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 13173.07it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
# =========================================================
# LOSS / OPTIMIZER
# =========================================================
class_counts = train_df[LABEL_COL].value_counts().sort_index().values
class_weights = torch.tensor(
    [len(train_df) / (2 * class_counts[0]), len(train_df) / (2 * class_counts[1])],
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

use_amp = (device.type == "xpu")

In [26]:
# =========================================================
# TRAIN FUNCTION
# =========================================================
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in loader:
        images = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        if use_amp:
            with torch.autocast(device_type="xpu", dtype=torch.float16):
                outputs = model(images, input_ids, attention_mask)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images, input_ids, attention_mask)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    return epoch_loss, epoch_acc

In [27]:
# =========================================================
# EVAL FUNCTION
# =========================================================
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_labels = []
    all_preds = []

    for batch in loader:
        images = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        if use_amp:
            with torch.autocast(device_type="xpu", dtype=torch.float16):
                outputs = model(images, input_ids, attention_mask)
                loss = criterion(outputs, labels)
        else:
            outputs = model(images, input_ids, attention_mask)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples
    return epoch_loss, epoch_acc, all_labels, all_preds

In [28]:
# =========================================================
# TRAIN LOOP
# =========================================================
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "label_to_id": label_to_id,
                "id_to_label": id_to_label,
                "text_model_name": TEXT_MODEL_NAME
            },
            MODEL_PATH
        )
        print("Best multimodal model saved.")

    scheduler.step()

print("\nTraining finished.")
print("Best validation accuracy:", best_val_acc)


Epoch 1/6
Train Loss: 0.6983 | Train Acc: 0.4764
Val   Loss: 0.6893 | Val   Acc: 0.5473
Best multimodal model saved.

Epoch 2/6
Train Loss: 0.6884 | Train Acc: 0.5393
Val   Loss: 0.6836 | Val   Acc: 0.5878
Best multimodal model saved.

Epoch 3/6
Train Loss: 0.6602 | Train Acc: 0.6382
Val   Loss: 0.6814 | Val   Acc: 0.5270

Epoch 4/6
Train Loss: 0.5883 | Train Acc: 0.7528
Val   Loss: 0.6829 | Val   Acc: 0.5676

Epoch 5/6
Train Loss: 0.4704 | Train Acc: 0.8787
Val   Loss: 0.7044 | Val   Acc: 0.5878

Epoch 6/6
Train Loss: 0.3888 | Train Acc: 0.9169
Val   Loss: 0.7284 | Val   Acc: 0.6216
Best multimodal model saved.

Training finished.
Best validation accuracy: 0.6216216216216216


In [29]:
# =========================================================
# TEST THE TRAINED MULTIMODAL MODEL
# =========================================================
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_acc, test_labels, test_preds = evaluate(model, test_loader)

print("\nTest results")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_preds))

print("\nClassification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=["non_offensive", "offensive"],
    digits=4
))


Test results
Test Loss: 0.6799
Test Acc : 0.6376

Confusion Matrix:
[[71 20]
 [34 24]]

Classification Report:
               precision    recall  f1-score   support

non_offensive     0.6762    0.7802    0.7245        91
    offensive     0.5455    0.4138    0.4706        58

     accuracy                         0.6376       149
    macro avg     0.6108    0.5970    0.5975       149
 weighted avg     0.6253    0.6376    0.6257       149



In [30]:
# =========================================================
# TEST A SINGLE MEME
# =========================================================
from PIL import Image

single_image_path = IMAGE_DIR / "LJ3r8Gy.jpg.png"
single_text = "OFFICIAL BERNIE SANDERS DRINKING GAME. TAKE A SHOT EVERY TIME HE SAYS WALL STREET"

model.eval()

image = Image.open(single_image_path).convert("RGB")
image = eval_transform(image).unsqueeze(0).to(device)

encoded = tokenizer(
    single_text,
    padding="max_length",
    truncation=True,
    max_length=MAX_TEXT_LEN,
    return_tensors="pt"
)

input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

with torch.no_grad():
    if use_amp:
        with torch.autocast(device_type="xpu", dtype=torch.float16):
            outputs = model(image, input_ids, attention_mask)
    else:
        outputs = model(image, input_ids, attention_mask)

    pred = outputs.argmax(dim=1).item()

print("Predicted label:", id_to_label[pred])

Predicted label: non_offensive


In [31]:
# =========================================================
# TEST ONE SAMPLE FROM TEST CSV
# =========================================================
row = test_df.iloc[0]

image_path = IMAGE_DIR / row["image_name"]
text = row["sentence"]
true_label = row["label"]

image = Image.open(image_path).convert("RGB")
image = eval_transform(image).unsqueeze(0).to(device)

encoded = tokenizer(
    str(text),
    padding="max_length",
    truncation=True,
    max_length=MAX_TEXT_LEN,
    return_tensors="pt"
)

input_ids = encoded["input_ids"].to(device)
attention_mask = encoded["attention_mask"].to(device)

model.eval()
with torch.no_grad():
    if use_amp:
        with torch.autocast(device_type="xpu", dtype=torch.float16):
            outputs = model(image, input_ids, attention_mask)
    else:
        outputs = model(image, input_ids, attention_mask)

    pred = outputs.argmax(dim=1).item()

print("Image name      :", row["image_name"])
print("Sentence        :", row["sentence"])
print("True label      :", id_to_label[int(true_label)])
print("Predicted label :", id_to_label[pred])

Image name      : jyxHhiB.png
Sentence        : 3 hrs Black nurse in Connecticut asked me if Trump was bringing back slavery in earnest . Chuck : No ma am . I know some real raaaacists but we do n't ever talk about bringing back slavery . That 's not on the agenda . Nurse : That 's good . What do y'all talk about ? Chuck : Mostly we do n't want your menfolk having raping our women , mugging us , or killing us . We also want you to stop having kids we got ta pay for Nurse : yeah we got to stop doing that . Well I liked Donald on the Apprentice . Ill vote for him . There are too many Puerto Ricans in this country . They the ones you got ta watch Why ca n't we have a conversation like this on race ? 
True label      : offensive
Predicted label : non_offensive
